# v2-03 · Unitary vs. general-CPTP quasi-inverse on Stinespring-realised channels

**What changed vs. `CPTP_Aer_IBM_Flow.ipynb`**

1. The draft-7 hardware run gave $D\approx0.9$–$0.95$ for *every two-qubit circuit* (bit flip, phase flip, bit–phase flip, amplitude damping) while every three-or-more-qubit circuit (GAD, mixed unitary, tetrahedron, and the CPTP branches that add an ancilla) was close to the simulator. Noise does not do that; a mapping/readout problem does. This notebook adds **control circuits** (bare $|+\rangle$, channel without recovery), records which physical qubit each circuit measured, and runs a **noise-model simulation of the same device first**. Do not submit to the QPU until the noise-model column agrees with the ideal column to within ~0.1 for every row.
2. The unitary quasi-inverse is now computed with the exact closed form (`quasi_inverse`) and cross-checked against the previous BFGS optimiser.
3. The CPTP recovery (cvxpy SDP) is kept, but its Choi-matrix convention is now verified numerically against the Kraus → affine map before use.
4. Standard errors, metadata and job IDs recorded; counts exported.

**What to send back:** `results_cptp_<backend>.json` and the printed comparison table (ideal Aer / noisy Aer / QPU).


In [ ]:
EXTRA_PKGS = ['cvxpy', 'clarabel', 'scipy']
# Colab / fresh environment: install once (comment out if already installed)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "qiskit>=1.4", "qiskit-aer", "qiskit-ibm-runtime>=0.40", "matplotlib", "pandas"] + EXTRA_PKGS,
               check=False)


## Shared, verified MSTD / quasi-inverse routines (identical in all v2 notebooks)

In [ ]:
# =============================================================================
#  MSTD quasi-inverse: shared, verified routines  (v2, 2026-11-05)
#  This block is pasted verbatim into every v2 notebook. Do not edit one copy
#  without editing the others.
#
#  Conventions (identical to the paper):
#    rho(r) = (I + r.sigma)/2,   channel  r -> M r + c
#    D^2(M,c) = [Tr(M M^T) - 2 Tr M + 3]/20 + |c|^2/4           (Bloch-ball MSTD)
#    V(x) = x0 I + i (x1 X + x2 Y + x3 Z),   |x| = 1             (unitary recovery)
#    V rho(r) V^dag = rho(R_V r),   V o E  ->  (R_V M, R_V c)
#
#  Key identity used for the quasi-inverse:
#    D^2(R M, R c) = D^2(M,c) - (1/10)[Tr(R M) - Tr M]           (R orthogonal)
#  so maximising the MSTD reduction  Delta D^2 = (1/10)[Tr(R_V M) - Tr M]
#  is the same as maximising Tr(R_V M), which is an exact quadratic form in x:
#    Delta D^2(x) = (1/5) x^T Qhat x,
#    Qhat_00 = 0,  Qhat_vv = (M + M^T)/2 - Tr(M) I_3,
#    Qhat_01 = -(M_23 - M_32)/2, Qhat_02 = -(M_31 - M_13)/2, Qhat_03 = -(M_12 - M_21)/2
#  (indices 1,2,3 = x,y,z).  The optimum is the top eigenvector of Qhat and
#  Delta D^2_max = lambda_max(Qhat)/5.  Because the identity (x = e_0) gives
#  x^T Qhat x = 0, lambda_max >= 0 always, and lambda_max = 0 means "do nothing".
#
#  NOTE: the expression in draft 7 of the paper (prefactor 2/5, diagonal block
#  (M+M^T)/2 - Tr(M)/2 I, off-diagonals -1/4 (...)) is NOT the quadratic form of
#  Delta D^2; self_test() below checks the one used here against brute force.
# =============================================================================
import json, math, time, datetime
import numpy as np

I2 = np.eye(2, dtype=complex)
PX = np.array([[0, 1], [1, 0]], dtype=complex)
PY = np.array([[0, -1j], [1j, 0]], dtype=complex)
PZ = np.array([[1, 0], [0, -1]], dtype=complex)
PAULIS = [PX, PY, PZ]


def mstd(M, c):
    """Bloch-ball mean-square trace distance of the affine channel (M, c)."""
    M = np.asarray(M, float); c = np.asarray(c, float).reshape(3)
    return float((np.trace(M @ M.T) - 2.0 * np.trace(M) + 3.0) / 20.0 + c @ c / 4.0)


def unitary_from_x(x):
    """V = x0 I + i(x1 X + x2 Y + x3 Z), normalised."""
    x = np.asarray(x, float); x = x / np.linalg.norm(x)
    return x[0] * I2 + 1j * (x[1] * PX + x[2] * PY + x[3] * PZ)


def rotation_from_unitary(V):
    """SO(3) matrix R_V with V rho(r) V^dag = rho(R_V r)."""
    return np.array([[0.5 * np.trace(si @ V @ sj @ V.conj().T).real for sj in PAULIS] for si in PAULIS])


def build_Qhat(M):
    """Exact 4x4 symmetric matrix with Delta D^2(x) = (1/5) x^T Qhat x."""
    M = np.asarray(M, float)
    Q = np.zeros((4, 4))
    Q[1:, 1:] = 0.5 * (M + M.T) - np.trace(M) * np.eye(3)
    Q[0, 1] = Q[1, 0] = -(M[1, 2] - M[2, 1]) / 2.0
    Q[0, 2] = Q[2, 0] = -(M[2, 0] - M[0, 2]) / 2.0
    Q[0, 3] = Q[3, 0] = -(M[0, 1] - M[1, 0]) / 2.0
    return Q


def quasi_inverse(M, c, min_gain=1e-4):
    """Optimal unitary quasi-inverse of the affine channel (M, c).

    Returns dict with x, V, R, Q, lambda_max, D2_raw, D2_corrected, delta_D2 (= delta_D2_predicted).
    If the best achievable reduction is below `min_gain` the identity is returned (nothing to correct).
    """
    M = np.asarray(M, float); c = np.asarray(c, float).reshape(3)
    Q = build_Qhat(M)
    evals, evecs = np.linalg.eigh(Q)
    lam = float(evals[-1]); x = np.real(evecs[:, -1]); x /= np.linalg.norm(x)
    if lam / 5.0 <= min_gain:           # channel is already best left alone
        lam = 0.0; x = np.array([1.0, 0.0, 0.0, 0.0])
    if x[0] < 0: x = -x                 # fix the global-phase sign convention
    V = unitary_from_x(x); R = rotation_from_unitary(V)
    D2_raw = mstd(M, c); D2_qi = mstd(R @ M, R @ c)
    out = dict(x=x, V=V, R=R, Q=Q, lambda_max=lam, D2_raw=D2_raw,
               D2_corrected=D2_qi, delta_D2=D2_raw - D2_qi, delta_D2_predicted=lam / 5.0)
    # internal consistency: the closed form must reproduce the direct evaluation
    assert abs(out["delta_D2"] - out["delta_D2_predicted"]) < 1e-9, "Qhat inconsistent with direct MSTD"
    assert out["delta_D2"] <= D2_raw + 1e-12, "reduction larger than the raw MSTD (impossible)"
    assert np.linalg.norm(V.conj().T @ V - I2) < 1e-12
    return out


def affine_from_tomography(r0, r1, rp, ry):
    """(M, c) from the Bloch vectors of the outputs for inputs |0>,|1>,|+>,|+y>."""
    r0, r1, rp, ry = (np.asarray(v, float) for v in (r0, r1, rp, ry))
    c = 0.5 * (r0 + r1)
    M = np.column_stack([rp - c, ry - c, 0.5 * (r0 - r1)])
    return M, c


def affine_from_pairs(r_in, r_out):
    """Least-squares (M, c) from any set of (input, output) Bloch vectors (>= 4 inputs)."""
    r_in = np.asarray(r_in, float); r_out = np.asarray(r_out, float)
    A = np.hstack([r_in, np.ones((len(r_in), 1))])           # [r | 1]
    sol, *_ = np.linalg.lstsq(A, r_out, rcond=None)          # sol^T = [M | c]
    return sol[:3].T, sol[3]


def expectation_and_se(counts, key0="0", key1="1"):
    """<P> = (n0 - n1)/N and its binomial standard error."""
    n0 = counts.get(key0, 0); n1 = counts.get(key1, 0); N = n0 + n1
    if N == 0: return float("nan"), float("nan")
    p = n0 / N; e = 2 * p - 1
    se = 2.0 * math.sqrt(max(p * (1 - p), 0.25 / N) / N)     # floor avoids se = 0 at p in {0,1}
    return e, se


def trace_distance(r1, r2):
    return 0.5 * float(np.linalg.norm(np.asarray(r1, float) - np.asarray(r2, float)))


def trace_distance_se(r1, r2, se2):
    """Propagated 1-sigma error of D(r1, r2) when r2 has component errors se2."""
    d = np.asarray(r1, float) - np.asarray(r2, float); n = np.linalg.norm(d)
    if n < 1e-12: return 0.5 * float(np.linalg.norm(se2))
    return 0.5 * float(math.sqrt(np.sum((d / n) ** 2 * np.asarray(se2, float) ** 2)))


def sampled_mstd(M, c, n=20000, seed=0):
    """Monte-Carlo D^2 over the uniform Bloch ball (cross-check of the closed form)."""
    rng = np.random.default_rng(seed)
    v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1)[:, None]
    r = v * rng.uniform(0, 1, n)[:, None] ** (1 / 3)
    d = r - (r @ np.asarray(M, float).T + np.asarray(c, float))
    return float(np.mean(np.sum(d * d, axis=1)) / 4.0)


def backend_metadata(backend, qubits):
    """Backend name, dt, and T1/T2/readout/gate errors for the physical qubits used."""
    meta = {"backend": getattr(backend, "name", str(backend)), "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "qubits": list(qubits)}
    try:
        meta["dt_seconds"] = float(backend.dt) if backend.dt is not None else None
    except Exception:
        meta["dt_seconds"] = None
    try:
        props = backend.properties()
        meta["calibration"] = {}
        for q in qubits:
            meta["calibration"][int(q)] = {
                "T1_us": props.t1(q) * 1e6, "T2_us": props.t2(q) * 1e6,
                "readout_error": props.readout_error(q),
                "sx_error": props.gate_error("sx", [q]) if "sx" in {g.gate for g in props.gates} else None}
    except Exception as exc:
        meta["calibration_error"] = str(exc)
    return meta


class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.ndarray):
            return o.tolist() if not np.iscomplexobj(o) else {"re": o.real.tolist(), "im": o.imag.tolist()}
        if isinstance(o, (np.floating, np.integer)): return o.item()
        if isinstance(o, complex): return {"re": o.real, "im": o.imag}
        return super().default(o)


def save_results(obj, filename):
    with open(filename, "w") as fh:
        json.dump(obj, fh, cls=NpEncoder, indent=1)
    print("saved", filename)
    try:                                   # Colab: trigger a download so the file can be sent back
        from google.colab import files
        files.download(filename)
    except Exception:
        pass


def self_test(n_random=200, seed=1):
    """Check Qhat against brute-force optimisation on random channels and on known cases."""
    rng = np.random.default_rng(seed)
    # exact quadratic form on random affine maps
    for _ in range(n_random):
        M = rng.normal(size=(3, 3)) * 0.5; c = rng.normal(size=3) * 0.1
        Q = build_Qhat(M)
        for _ in range(3):
            x = rng.normal(size=4); x /= np.linalg.norm(x)
            R = rotation_from_unitary(unitary_from_x(x))
            dd = mstd(M, c) - mstd(R @ M, R @ c)
            assert abs(dd - x @ Q @ x / 5.0) < 1e-10, "Qhat is not the quadratic form of Delta D^2"
        qi = quasi_inverse(M, c)
        best = min(mstd(rotation_from_unitary(unitary_from_x(v)) @ M, rotation_from_unitary(unitary_from_x(v)) @ c)
                   for v in rng.normal(size=(3000, 4)))
        assert qi["D2_corrected"] <= best + 1e-9, "eigenvector is not the optimum"
    # bit flip p=0.55: M=diag(1,-0.1,-0.1) -> V = X, Delta D^2 = 0.04
    qi = quasi_inverse(np.diag([1, -0.1, -0.1]), np.zeros(3))
    assert abs(qi["delta_D2"] - 0.04) < 1e-12 and abs(abs(qi["x"][1]) - 1) < 1e-9
    # near-identity channel -> V = identity
    qi = quasi_inverse(0.97 * np.eye(3), np.array([0, 0, 0.03]))
    assert qi["lambda_max"] == 0.0 and qi["x"][0] == 1.0
    # sampled MSTD agrees with closed form
    M = rng.normal(size=(3, 3)) * 0.4; c = rng.normal(size=3) * 0.1
    assert abs(sampled_mstd(M, c, 200000) - mstd(M, c)) < 2e-3
    print("mstd_qi_common self-test passed")



self_test()


## Backend

In [ ]:
# ---------------------------------------------------------------------------
# Backend selection.  NEVER paste a token into this notebook (two earlier
# notebooks had tokens committed in plain text -- those tokens must be revoked).
# Run ONCE in a separate scratch cell, then delete that cell:
#   from qiskit_ibm_runtime import QiskitRuntimeService
#   QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...", overwrite=True)
# Alternatively set the environment variable IBM_QUANTUM_TOKEN (Colab: use userdata secrets).
# ---------------------------------------------------------------------------
import os
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.transpiler import generate_preset_pass_manager

USE_SIMULATOR = False            # <-- set False for the real QPU run
BACKEND_NAME  = "ibm_fez"       # the paper's primary device; keep all experiments on one device
SHOTS         = 4096

if USE_SIMULATOR:
    from qiskit_aer import AerSimulator
    from qiskit_ibm_runtime.fake_provider import FakeFez
    from qiskit_aer.primitives import SamplerV2 as Sampler
    fake = FakeFez()
    backend = AerSimulator.from_backend(fake)        # noise model + coupling map of ibm_fez
    backend.name = "aer(" + fake.name + ")"
    cal_backend = fake                               # properties() for metadata
    sampler = Sampler.from_backend(fake)
else:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
    token = os.environ.get("IBM_QUANTUM_TOKEN")
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token) if token else QiskitRuntimeService()
    backend = service.backend(BACKEND_NAME)
    cal_backend = backend
    sampler = Sampler(mode=backend)

print("Backend:", backend.name, "| simulator:", USE_SIMULATOR, "| shots:", SHOTS)

JOB_LOG = []   # every submitted job: id, purpose, number of circuits

def run_batch(circuits, purpose, physical_qubits=None, optimization_level=1):
    """Transpile + run one list of circuits; returns a list of counts dicts (same order)."""
    target = cal_backend if USE_SIMULATOR else backend
    pm = generate_preset_pass_manager(backend=target, optimization_level=optimization_level,
                                      initial_layout=physical_qubits)
    isa = pm.run(circuits)
    # sanity: delays must survive transpilation
    for q0, q1 in zip(circuits, isa):
        n_delay0 = sum(1 for inst in q0.data if inst.operation.name == "delay")
        n_delay1 = sum(1 for inst in q1.data if inst.operation.name == "delay")
        assert n_delay1 >= n_delay0, "transpiler removed a delay -- lower optimization_level"
    job = sampler.run(isa, shots=SHOTS)
    jid = job.job_id() if hasattr(job, "job_id") else "simulator"
    JOB_LOG.append({"job_id": jid, "purpose": purpose, "n_circuits": len(isa),
                    "depths": [c.depth() for c in isa], "two_qubit_gates": [c.num_nonlocal_gates() for c in isa],
                    "physical_qubits": [sorted({q._index if hasattr(q, "_index") else backend_index(c, q)
                                                for inst in c.data for q in inst.qubits if inst.operation.name == "measure"})
                                        for c in isa]})
    print(f"[{purpose}] job {jid}: {len(isa)} circuits submitted")
    res = job.result()
    out = []
    for pub in res:
        data = pub.data
        reg = [k for k in data.keys()][0]
        out.append(getattr(data, reg).get_counts())
    return out

def backend_index(circ, q):
    return circ.find_bit(q).index


## Tomography helpers

In [ ]:
# ---------------------------------------------------------------------------
# State preparation / measurement helpers (single logical qubit, register "meas")
# ---------------------------------------------------------------------------
PAULI_STATES = {            # name: (prep gates, Bloch vector)
    "0":  ([],            np.array([0, 0, 1.0])),
    "1":  (["x"],         np.array([0, 0, -1.0])),
    "+":  (["h"],         np.array([1.0, 0, 0])),
    "-":  (["x", "h"],    np.array([-1.0, 0, 0])),
    "+y": (["h", "s"],    np.array([0, 1.0, 0])),
    "-y": (["h", "sdg"],  np.array([0, -1.0, 0])),
}
CHAR_INPUTS = ["0", "1", "+", "+y"]      # minimal set that determines (M, c)
VAL_INPUTS  = list(PAULI_STATES)         # all six Pauli eigenstates for validation

def prepare(qc, q, state):
    for g in PAULI_STATES[state][0]:
        getattr(qc, g)(q)

def rotate_to_z(qc, q, basis):
    if basis == "X":   qc.h(q)
    elif basis == "Y": qc.sdg(q); qc.h(q)       # (H S^dag)^dag Z (H S^dag) = Y
    elif basis != "Z": raise ValueError(basis)

def bloch_from_counts(counts_xyz):
    """[counts_X, counts_Y, counts_Z] -> (r, se)"""
    vals = [expectation_and_se(c) for c in counts_xyz]
    return np.array([v[0] for v in vals]), np.array([v[1] for v in vals])


## Channels

In [ ]:
# ---- channels (identical parameters to draft 7) ----
def bit_flip(p):        return [np.sqrt(1-p)*I2, np.sqrt(p)*PX]
def phase_flip(p):      return [np.sqrt(1-p)*I2, np.sqrt(p)*PZ]
def bit_phase_flip(p):  return [np.sqrt(1-p)*I2, np.sqrt(p)*PY]
def amp_damp(g):        return [np.array([[1,0],[0,np.sqrt(1-g)]],complex), np.array([[0,np.sqrt(g)],[0,0]],complex)]
def gad(g, p):          return [np.sqrt(p)*np.array([[1,0],[0,np.sqrt(1-g)]],complex), np.sqrt(p)*np.array([[0,np.sqrt(g)],[0,0]],complex),
                                np.sqrt(1-p)*np.array([[np.sqrt(1-g),0],[0,1]],complex), np.sqrt(1-p)*np.array([[0,0],[np.sqrt(g),0]],complex)]
def mixed_unitary(p):   return [np.sqrt(1-3*p)*I2, np.sqrt(p)*PX, np.sqrt(p)*PY, np.sqrt(p)*PZ]
def tetrahedron():
    vs = [np.array(v)/np.sqrt(3) for v in ([1,1,1],[1,-1,-1],[-1,1,-1],[-1,-1,1])]
    return [0.5*(v[0]*PX+v[1]*PY+v[2]*PZ) for v in vs]

CHANNELS = {"Bit flip": bit_flip(0.55), "Phase flip": phase_flip(0.55), "Bit-phase flip": bit_phase_flip(0.55),
            "Amplitude damping": amp_damp(0.20), "GAD": gad(0.20, 0.80), "Mixed unitary": mixed_unitary(0.10),
            "Tetrahedron": tetrahedron()}

def apply_kraus(ks, rho): return sum((K @ rho @ K.conj().T for K in ks), np.zeros((2,2),complex))
def bloch_of(rho):        return np.array([np.trace(rho @ P).real for P in PAULIS])
def kraus_to_affine(ks):
    c = bloch_of(apply_kraus(ks, I2/2)); M = np.zeros((3,3))
    for j, P in enumerate(PAULIS): M[:, j] = bloch_of(apply_kraus(ks, (I2+P)/2)) - c
    return M, c
def compose(after, before): return [A @ B for A in after for B in before]

for name, ks in CHANNELS.items():
    comp = sum((K.conj().T @ K for K in ks), np.zeros((2,2),complex)); assert np.linalg.norm(comp - I2) < 1e-9, name
    M, c = kraus_to_affine(ks); print(f"{name:18s} rank={len(ks)}  D2={mstd(M,c):.4f}  c={np.round(c,3)}")


## Unitary quasi-inverse

In [ ]:
# ---- unitary quasi-inverse: closed form, cross-checked against BFGS ----
from scipy.optimize import minimize
UNITARY = {}
for name, ks in CHANNELS.items():
    M, c = kraus_to_affine(ks); qi = quasi_inverse(M, c)
    def obj(q):
        R = rotation_from_unitary(unitary_from_x(q)); return mstd(R @ M, R @ c)
    rng = np.random.default_rng(0)
    bfgs = min((minimize(obj, q0, method="BFGS").fun for q0 in [np.array([1.,0,0,0])] + list(rng.normal(size=(15,4)))))
    assert qi["D2_corrected"] <= bfgs + 1e-6, f"{name}: closed form worse than BFGS?!"
    UNITARY[name] = qi
    print(f"{name:18s} D2 {qi['D2_raw']:.4f} -> {qi['D2_corrected']:.4f} (BFGS {bfgs:.4f})  x={np.round(qi['x'],3)}")


## General CPTP quasi-inverse (SDP)

In [ ]:
# ---- general CPTP recovery by SDP over the (normalised) Choi matrix ----
import cvxpy as cp
from scipy.linalg import null_space, polar

def choi_normalised(ks):
    """J = (R (x) id)(|Phi><Phi|), output first, Tr J = 1."""
    phi = np.zeros(4, complex); phi[0] = phi[3] = 1/np.sqrt(2)
    rho = np.outer(phi, phi.conj()); J = np.zeros((4,4), complex)
    for K in ks: KI = np.kron(K, I2); J += KI @ rho @ KI.conj().T
    return J

def affine_from_choi(J):
    A = np.array([[np.trace(J @ np.kron(PAULIS[i], PAULIS[j].T)).real for j in range(3)] for i in range(3)])
    b = np.array([np.trace(J @ np.kron(PAULIS[i], I2)).real for i in range(3)])
    return A, b

# convention check: Choi -> affine must agree with Kraus -> affine for every channel
for name, ks in CHANNELS.items():
    A, b = affine_from_choi(choi_normalised(ks)); M, c = kraus_to_affine(ks)
    assert np.allclose(A, M, atol=1e-9) and np.allclose(b, c, atol=1e-9), f"Choi convention mismatch for {name}"
print("Choi convention verified")

def cptp_recovery(M, c):
    A = cp.Variable((3,3)); b = cp.Variable(3)
    J = cp.Constant(np.eye(4, dtype=complex)/4)
    for i in range(3): J = J + b[i] * cp.Constant(np.kron(PAULIS[i], I2)/4)
    for i in range(3):
        for j in range(3): J = J + A[i,j] * cp.Constant(np.kron(PAULIS[i], PAULIS[j].T)/4)
    cons = [J >> 0]
    for a in range(2):
        for bb in range(2):
            el = sum(J[2*u+a, 2*u+bb] for u in range(2))          # Tr_out J = I/2
            cons += [cp.real(el) == (0.5 if a == bb else 0.0), cp.imag(el) == 0.0]
    prob = cp.Problem(cp.Minimize(cp.sum_squares(A @ M - np.eye(3))/20 + cp.sum_squares(A @ c + b)/4), cons)
    for solver in ("CLARABEL", "SCS"):
        try:
            prob.solve(solver=solver)
            if prob.status in ("optimal", "optimal_inaccurate"): break
        except Exception as e: last = e
    assert prob.status in ("optimal", "optimal_inaccurate"), prob.status
    Jv = np.asarray(J.value); Jv = (Jv + Jv.conj().T)/2
    vals, vecs = np.linalg.eigh(Jv)
    ks = [np.sqrt(2*l) * v.reshape(2,2) for l, v in zip(vals, vecs.T) if l > 1e-7]
    comp = sum((K.conj().T @ K for K in ks), np.zeros((2,2),complex))
    return dict(A=np.asarray(A.value), b=np.asarray(b.value).ravel(), J=Jv, kraus=ks, D2=float(prob.value),
                tp_error=float(np.linalg.norm(comp - I2)))

CPTP = {}
for name, ks in CHANNELS.items():
    M, c = kraus_to_affine(ks); sol = cptp_recovery(M, c)
    # verify by direct composition of Kraus operators
    Mr, cr = kraus_to_affine(compose(sol["kraus"], ks)); d2_direct = mstd(Mr, cr)
    assert abs(d2_direct - sol["D2"]) < 5e-3, f"{name}: SDP objective {sol['D2']:.4f} vs direct {d2_direct:.4f}"
    CPTP[name] = sol
    print(f"{name:18s} unitary D2_QI={UNITARY[name]['D2_corrected']:.4f}   CPTP D2={d2_direct:.4f}   rank={len(sol['kraus'])}  TP err={sol['tp_error']:.1e}")


## Circuits

In [ ]:
# ---- Stinespring dilation and experiment circuits (system = qubit 0, environment qubits after it) ----
def stinespring(ks):
    r = len(ks); dE = 1 << int(np.ceil(np.log2(max(1, r)))); W = np.zeros((2*dE, 2), complex)
    for j, K in enumerate(ks): W[2*j:2*j+2, :] = K          # index = 2*env + sys  (sys = least significant = q[0])
    assert np.linalg.norm(W.conj().T @ W - I2) < 1e-5, "Kraus set is not trace preserving"
    W, _ = polar(W)                                           # nearest exact isometry (SDP output is TP only to ~1e-7)
    U = np.column_stack([W, null_space(W.conj().T)]); U, _ = polar(U)
    assert np.linalg.norm(U.conj().T @ U - np.eye(2*dE)) < 1e-10
    return U, int(np.log2(dE))

TARGET_STATE = "+"
def experiment_circuit(ks, branch, basis, V=None, rec_kraus=None):
    Ue, nE = stinespring(ks); nR = 0; Ur = None
    if branch == "cptp": Ur, nR = stinespring(rec_kraus)
    qr = QuantumRegister(1 + nE + nR, "q"); cr = ClassicalRegister(1, "meas"); qc = QuantumCircuit(qr, cr)
    prepare(qc, qr[0], TARGET_STATE); qc.barrier()
    if branch != "control":
        qc.append(UnitaryGate(Ue, label="U_E"), [qr[0]] + [qr[i] for i in range(1, 1+nE)]); qc.barrier()
    if branch == "unitary": qc.append(UnitaryGate(V, label="V_QI"), [qr[0]]); qc.barrier()
    if branch == "cptp":    qc.append(UnitaryGate(Ur, label="U_R"), [qr[0]] + [qr[i] for i in range(1+nE, 1+nE+nR)]); qc.barrier()
    rotate_to_z(qc, qr[0], basis); qc.measure(qr[0], cr[0])
    return qc

def batch():
    circuits, index = [], []
    for b in "XYZ":
        circuits.append(experiment_circuit(CHANNELS["Bit flip"], "control", b)); index.append(("control |+>", "control", b))
    for name, ks in CHANNELS.items():
        for branch, kw in (("none", {}), ("unitary", {"V": UNITARY[name]["V"]}), ("cptp", {"rec_kraus": CPTP[name]["kraus"]})):
            for b in "XYZ":
                circuits.append(experiment_circuit(ks, branch, b, **kw)); index.append((name, branch, b))
    return circuits, index

def analyse(counts, index, platform):
    import pandas as pd
    byk = {}
    for (name, branch, b), cnt in zip(index, counts): byk.setdefault((name, branch), {})[b] = cnt
    rows = []
    for (name, branch), d in byk.items():
        r, se = bloch_from_counts([d[b] for b in "XYZ"])
        rows.append(dict(platform=platform, channel=name, branch=branch, D=trace_distance(PAULI_STATES[TARGET_STATE][1], r),
                         D_se=trace_distance_se(PAULI_STATES[TARGET_STATE][1], r, se), r=np.round(r, 3)))
    return pd.DataFrame(rows)

CIRCUITS, INDEX = batch()
print(len(CIRCUITS), "circuits; qubit counts:", sorted({c.num_qubits for c in CIRCUITS}))


## Stage A/B - ideal and noise-model simulation

In [ ]:
# ---- Stage A: ideal simulator (must match the SDP/closed-form predictions) ----
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler import generate_preset_pass_manager
ideal = AerSimulator(); pm0 = generate_preset_pass_manager(backend=ideal, optimization_level=1)
res = AerSampler().run(pm0.run(CIRCUITS), shots=SHOTS).result()
counts_ideal = [getattr(p.data, list(p.data.keys())[0]).get_counts() for p in res]
T_IDEAL = analyse(counts_ideal, INDEX, "Aer ideal")

# ---- Stage B: noise model of the target device (same transpilation path as the QPU) ----
counts_noisy = run_batch(CIRCUITS, "noise-model validation") if USE_SIMULATOR else None
# (when USE_SIMULATOR is False, run_batch goes to the real QPU; see next cell)


## Stage C - device run and comparison table

In [ ]:
import pandas as pd
if USE_SIMULATOR:
    T_DEV = analyse(counts_noisy, INDEX, "Aer(" + cal_backend.name + ")")
else:
    counts_qpu = run_batch(CIRCUITS, "QPU run")
    T_DEV = analyse(counts_qpu, INDEX, backend.name)

T = T_IDEAL.merge(T_DEV, on=["channel", "branch"], suffixes=("_ideal", "_dev"))
T["gap"] = T.D_dev - T.D_ideal
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
display(T[["channel", "branch", "D_ideal", "D_dev", "D_se_dev", "gap", "r_ideal", "r_dev"]])
bad = T[T.gap.abs() > 0.15]
if len(bad):
    print("\n!!! rows whose device/noise-model result is far from the ideal simulator -- investigate before trusting:")
    display(bad[["channel", "branch", "D_ideal", "D_dev"]])
    print("physical qubits measured per circuit (from JOB_LOG):", JOB_LOG[-1]["physical_qubits"][:12], "...")
else:
    print("\nall rows within 0.15 of the ideal simulator")


## Figure

In [ ]:
import matplotlib.pyplot as plt
names = list(CHANNELS); x = np.arange(len(names)); w = 0.27
fig, ax = plt.subplots(figsize=(8.5, 3.8))
for i, (branch, lab) in enumerate((("none", "no recovery"), ("unitary", "unitary $V_{QI}$"), ("cptp", "general CPTP"))):
    t = T[T.branch == branch].set_index("channel").loc[names]
    ax.bar(x + (i-1)*w, t.D_dev, w, yerr=t.D_se_dev, capsize=2, label=lab)
    ax.plot(x + (i-1)*w, t.D_ideal, "k_", markersize=9, mew=1.5, label="ideal simulator" if i == 0 else None)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=25, ha="right"); ax.set_ylabel(r"trace distance to $|+\rangle$")
ax.grid(alpha=.3, axis="y"); ax.legend(ncol=2); plt.tight_layout(); plt.savefig("fig_cptp_v2.pdf"); plt.show()


## Export results (send this file back)

In [ ]:
meta = backend_metadata(cal_backend, [0, 1, 2]); meta["simulator"] = USE_SIMULATOR; meta["shots"] = SHOTS
results = {"experiment": "cptp_v2", "metadata": meta, "jobs": JOB_LOG,
           "unitary": {n: {k: v for k, v in q.items()} for n, q in UNITARY.items()},
           "cptp": {n: {"A": s["A"], "b": s["b"], "kraus": s["kraus"], "D2": s["D2"]} for n, s in CPTP.items()},
           "table": T.drop(columns=[c for c in T.columns if c.startswith("r_")]).to_dict(orient="records"),
           "index": INDEX, "counts_ideal": counts_ideal,
           "counts_device": counts_noisy if USE_SIMULATOR else counts_qpu}
save_results(results, f"results_cptp_{meta['backend'].replace('(', '_').replace(')', '')}.json")
